In [ ]:
import pandas as pd

roll_number = "1024170139"
categories = ["billing", "account", "general"]
last_two_digits = [int(d) for d in roll_number[-2:]]  # [4, 8]

fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing",
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account",
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general",
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing",
    },
]

personalized_entries = []
for digit in last_two_digits:
    category = categories[digit % 3]  # 4%3=1 -> "account", 8%3=2 -> "general"
    if digit == 0:
        entry = {
            "question": "how can i get a refund for a payment",
            "answer": "You can request a refund through the billing section.",
            "keywords": "refund payment billing money",
            "category": category,
        }
    elif digit == 2:
        entry = {
            "question": "where can i find the office address",
            "answer": "The office address is available on the Contact Us page.",
            "keywords": "office address location contact",
            "category": category,
        }
    else:
        entry = {
            "question": "how to update contact details",
            "answer": "Go to Account Settings to update your email or phone.",
            "keywords": "update contact details email phone",
            "category": category,
        }
    personalized_entries.append(entry)

all_entries = fixed_entries + personalized_entries
df = pd.DataFrame(all_entries)


def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()][
        ["question", "answer", "keywords", "category"]
    ]


personalized_category = personalized_entries[0]["category"]  # "account"
print("\nQ3: Entries belonging to category:", personalized_category)
print(same_category(personalized_category, df))


Q3: Entries belonging to category: account
                        question  \
1          how to reset password   
4  how to update contact details   

                                              answer  \
1                   Go to Settings > Reset Password.   
4  Go to Account Settings to update your email or...   

                             keywords category  
1                password reset login  account  
4  update contact details email phone  account  


In [ ]:
def score_query(query, df):
    """
    Takes a query and returns all matching entries
    ranked by confidence score.
    """
    query_words = set(re.findall(r'\b\w+\b', query.lower()))
    results = []
    for index, row in df.iterrows():
        keyword_words = set(
            re.findall(r'\b\w+\b', row["keywords"].lower())
        )
        question_words = set(
            re.findall(r'\b\w+\b', row["question"].lower())
        )
        searchable_words = keyword_words | question_words
        matches = query_words & searchable_words
        if len(query_words) > 0:
            score = len(matches) / len(query_words)
        else:
            score = 0
        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })
    results.sort(key=lambda x: x["score"], reverse=True)
    return results
query = "how can I pay my fee"
results = score_query(query, df)
print("\nQ2 Results:")
for result in results:
    print(result)


Q2 Results:
{'question': 'how can i pay the fee', 'answer': 'You can pay via UPI, card, or net banking.', 'category': 'billing', 'score': 0.8333333333333334}
{'question': 'what is the annual fee', 'answer': 'The annual fee is Rs 500.', 'category': 'billing', 'score': 0.16666666666666666}
{'question': 'how to reset password', 'answer': 'Go to Settings > Reset Password.', 'category': 'account', 'score': 0.16666666666666666}


In [ ]:
def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()][
        ["question", "answer", "keywords", "category"]
    ]
personalized_category = personalized_entries[0]["category"]
print("\nQ3: Entries belonging to category:",
      personalized_category)
print(same_category(personalized_category, df))


Q3: Entries belonging to category: account
                        question  \
1          how to reset password   
4  how to update contact details   

                                              answer  \
1                   Go to Settings > Reset Password.   
4  Go to Account Settings to update your email or...   

                             keywords category  
1                password reset login  account  
4  update contact details email phone  account  


In [ ]:
print("\nAvailable entries:")
for i, question in enumerate(df["question"]):
    print(i, "-", question)
choice = int(input("\nChoose an entry number: "))
new_keyword = input("Enter a new keyword: ").strip()
old_keywords = df.loc[choice, "keywords"]
df.loc[choice, "keywords"] = old_keywords + " " + new_keyword
filename = roll_number + "_faq_data.csv"
df.to_csv(filename, index=False)
print("\nUpdated DataFrame:")
print(df)
print("\nSaved successfully as:", filename)


Available entries:
0 - what is the annual fee
1 - how to reset password
2 - what are your working hours
3 - how can i pay the fee
4 - how to update contact details
5 - how to update contact details

Choose an entry number: 5
Enter a new keyword: 2

Updated DataFrame:
                        question  \
0         what is the annual fee   
1          how to reset password   
2    what are your working hours   
3          how can i pay the fee   
4  how to update contact details   
5  how to update contact details   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  Go to Account Settings to update your email or...   
5  Go to Account Settings to update your email or...   

                               keywords category  
0                 fee cost price c

In [ ]:
category_counts = df.groupby("category").size()
print("\nQ5: Number of FAQ entries per category:")
print(category_counts)


Q5: Number of FAQ entries per category:
category
account    2
billing    2
general    2
dtype: int64


In [ ]:
def score_query_with_ties(query, df):
    query_words = set(re.findall(r'\b\w+\b', query.lower()))
    results = []
    for index, row in df.iterrows():
        keyword_words = set(
            re.findall(r'\b\w+\b', row["keywords"].lower())
        )
        question_words = set(
            re.findall(r'\b\w+\b', row["question"].lower())
        )
        searchable_words = keyword_words | question_words
        matches = query_words & searchable_words
        if len(query_words) > 0:
            score = len(matches) / len(query_words)
        else:
            score = 0
        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })
    if not results:
        print("No matching entries found.")
        return []
    results.sort(key=lambda x: x["score"], reverse=True)
    highest_score = results[0]["score"]
    best_matches = [
        result for result in results
        if result["score"] == highest_score
    ]
    return best_matches
print("\nQ6: Query producing a tie")
tie_query = "fee"
tie_results = score_query_with_ties(tie_query, df)
print("\nAll highest-scoring matches:")
for result in tie_results:
    print(result)
print("\nQ6: Query that does NOT produce a tie")
non_tie_query = "password"
non_tie_results = score_query_with_ties(non_tie_query, df)
for result in non_tie_results:
    print(result)


Q6: Query producing a tie

All highest-scoring matches:
{'question': 'what is the annual fee', 'answer': 'The annual fee is Rs 500.', 'category': 'billing', 'score': 1.0}
{'question': 'how can i pay the fee', 'answer': 'You can pay via UPI, card, or net banking.', 'category': 'billing', 'score': 1.0}

Q6: Query that does NOT produce a tie
{'question': 'how to reset password', 'answer': 'Go to Settings > Reset Password.', 'category': 'account', 'score': 1.0}
